In [1]:
# Cell 1: Basic Setup
from pydantic_ai.hooks.tool import ReactiveTool

In [2]:
# Cell 2: Create a Simple Tool
async def multiply(x: int) -> int:
    """Multiply a number by itself."""
    return x * x

# Create a reactive tool
tool = ReactiveTool(
    name="multiply", 
    function=multiply,
    description="Multiplies a number by itself"
)

In [3]:
# Cell 3: Watch State Changes
# Our current implementation uses Traitlets observe
def watch_state(change):
    print(f"State changed from {change.old} to {change.new}")

tool.state.observe(watch_state, names='state')

In [4]:
# Cell 4: Add Hooks (current implementation)
async def log_before(context):
    print(f"About to run with args: {context['args']}")

async def log_after(context):
    print(f"Got result: {context['result']}")

async def log_error(context):
    print(f"Error occurred: {context['error']}")

# Current hook assignment
tool.on.before = log_before
tool.on.after = log_after
tool.on.error = log_error

In [5]:
# Cell 5: Try it out
await tool(5)

State changed from ToolState.IDLE to ToolState.RUNNING
About to run with args: (5,)
State changed from ToolState.RUNNING to ToolState.COMPLETED
Got result: 25


25

In [6]:
# Cell 6: Try with error
try:
    await tool("not a number")
except TypeError as e:
    print("Caught expected error")

State changed from ToolState.COMPLETED to ToolState.RUNNING
About to run with args: ('not a number',)
State changed from ToolState.RUNNING to ToolState.ERROR
Error occurred: can't multiply sequence by non-int of type 'str'
Caught expected error


In [7]:
# Cell 7: Check current state
print(f"Tool state: {tool.state.state}")
print(f"Last result: {tool.state.result}")
print(f"Last error: {tool.state.error}")
print(f"Last context: {tool.state.context}")

Tool state: ToolState.ERROR
Last result: None
Last error: can't multiply sequence by non-int of type 'str'
Last context: {'tool': ReactiveTool(function=<function multiply at 0x1080ff920>, takes_ctx=False, max_retries=None, name='multiply', description='Multiplies a number by itself', prepare=None, docstring_format='auto', require_parameter_descriptions=False, strict=None, function_schema=FunctionSchema(function=<function multiply at 0x1080ff920>, description='Multiply a number by itself.', validator=<pydantic.plugin._schema_validator.PluggableSchemaValidator object at 0x107855800>, json_schema={'additionalProperties': False, 'properties': {'x': {'type': 'integer'}}, 'required': ['x'], 'type': 'object'}, takes_ctx=False, is_async=True, single_arg_name=None, positional_fields=[], var_positional_field=None)), 'args': ('not a number',), 'kwargs': {}}
